# Part 3: The datasets library

The `datasets` library loads a dataset from the Hub in one line and gives you splits, column types and fast `map` operations.

In [1]:
from review_data import load_reviews, small_sample

reviews = load_reviews()
reviews

c:\Users\student\AppData\Local\anaconda3\envs\aigc-5005-module09-hf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\student\AppData\Local\anaconda3\envs\aigc-5005-module09-hf\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\student\.cache\huggingface\hub\datasets--cornell-movie-review-data--rotten_tomatoes. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [2]:
print(reviews["train"].features)
print(reviews["train"][0])
print(reviews["train"].features["label"].names)

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}
{'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}
['neg', 'pos']


## Check the labels and the splits

Always look before you train. These splits are balanced, and no sentence appears in both the training and validation splits.

In [3]:
from collections import Counter

print(Counter(reviews["train"]["label"]))
print(len(set(reviews["train"]["text"]) & set(reviews["validation"]["text"])))

Counter({1: 4265, 0: 4265})
0


## A small labelled dataset

In practice you often have a few hundred labelled examples, not eight thousand. `small_sample` draws a balanced random sample with a seed, so you can repeat it.

In [4]:
texts, labels = small_sample(reviews["train"], 200, seed=42)
print(len(texts), sum(labels))
print(texts[0])

200 100
fear permeates the whole of stortelling , todd solondz' oftentimes funny , yet ultimately cowardly autocritique .


## Tokenizing with map

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
tokenized = reviews["validation"].map(
    lambda row: tokenizer(row["text"], truncation=True, max_length=64),
    batched=True,
)
print(tokenized.column_names)
print(max(len(tokenizer(text)["input_ids"]) for text in reviews["validation"]["text"]))

Map: 100%|██████████| 1066/1066 [00:00<00:00, 14285.53 examples/s]

['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
72


## Try it

1. Print five training sentences at random and read them. Would you label them the same way?
2. Find the longest sentence in the validation split. How many tokens is it?
3. Draw a sample of 50 with another seed. Is it still balanced?